# Day 13 — GAT: Graph Attention Networks

> Week 2 Graph Learning  
> 主线：**GCN / GraphSAGE → 为什么还需要 GAT → attention coefficient → softmax over neighbors → multi-head → PyG GATConv → Transformer 对照**

今天不追求把所有 attention 变体学完，只吃透经典 GAT。

---

## 今天必须拿下

1. 为什么 GCN / GraphSAGE 的邻居权重仍然不够灵活？
2. GAT 中 $e_{vu}$ 和 $α_{vu}$ 分别是什么？
3. 为什么 softmax 是 **对每个 target node 的 neighbors** 做？
4. attention coefficient 为什么和 sender / receiver 都有关？
5. 一层 GAT 的 input / intermediate / output shape。
6. multi-head attention 为什么存在？
7. `concat=True / False` 对 shape 的影响。
8. GAT 和 Transformer attention 的共同点与区别。
9. PyG `GATConv(x, edge_index)` 中 `x` / `edge_index` / trainable parameters 各做什么。
10. 能跑通一个 GAT node classification 小实验。

## 1. 从 GCN / GraphSAGE 走到 GAT

CS224W 这一部分的核心问题是：

> **不同 neighbor 对 target node 的重要性真的应该一样吗？**

### GCN / Mean GraphSAGE 的对照直觉

在课件这一页的简化写法里，可以把邻居权重理解成：

$$
\alpha_{vu} = \frac{1}{|N(v)|}
$$

即对于固定的 target node \(v\)，所有 \(u \in N(v)\) 被等权处理。

更一般地说，GCN 的权重由图结构 / degree normalization 规定，而不是根据当前 node features 动态学出来。

### GAT

GAT 把这个权重变成可学习的：

$$
h_v^{(l)}
=
\sigma\left(
\sum_{u\in N(v)}
\alpha_{vu}\,
W^{(l)}h_u^{(l-1)}
\right)
$$

其中：

$$
\alpha_{vu}
$$

表示 **node \(u\) 的 message 对 target node \(v\) 的重要性**。

GAT 的目标就是：

> **让模型自己学习不同 neighbors 对当前 target node 的不同重要性。**

## 必须回答 1

为什么 GAT 相比 GCN / Mean GraphSAGE 更灵活？

你的回答：GAT 更灵活，因为它能根据 node features 学习不同 neighbor 对 target node 的不同权重，而 GCN / Mean GraphSAGE 的 neighbor weighting rule 是预先规定的。

## 2. Step 1 — Feature Transformation

对每个 node：

$$
h_v \in \mathbb{R}^{d_{in}}
$$

先做：

$$
z_v = W h_v
$$

其中：

$$
W\in\mathbb{R}^{d_{out}\times d_{in}}
$$

shape：

```text
h_v        [d_in]
  ↓ W
z_v        [d_out]
```

全图：

```text
H          [N, d_in]
 ↓ Linear
Z          [N, d_out]
```

所有 node 共享同一个 `W`。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

H = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0],
    [1.0, 0.5, 0.0],
    [0.5, 1.0, 0.0],
    [0.0, 1.0, 1.0],
])

linear = nn.Linear(3, 4, bias=False)
Z = linear(H)

print("H shape:", H.shape)
print("Z shape:", Z.shape)
print(Z)

H shape: torch.Size([6, 3])
Z shape: torch.Size([6, 4])
tensor([[ 0.3061,  0.6469,  0.2279, -0.3155],
        [ 0.3439, -0.0100,  0.8480,  0.6099],
        [ 0.9206,  0.4039,  0.0580,  0.0783],
        [ 0.6810,  0.4671, -0.1116, -0.1726],
        [ 0.6999,  0.1387,  0.1985,  0.2901],
        [ 0.3439, -0.0100,  0.8480,  0.6099]], grad_fn=<MmBackward0>)


## 必须回答 2

如果：

```text
H.shape = [6, 3]
W: 3 → 4
```

那么：

```text
Z.shape = ?
```

为什么 GAT 要先做 $z_v=Wh_v$，再用 $z_u,z_v$ 计算 attention score，而不是直接用原始 $h_u,h_v$？

你的回答：[6, 4];虽然原始 node embeddings 本来维度一致，但 $W$ 不是单纯为了对齐维度，而是为了学习当前 GAT layer 所需的 feature transformation。模型先得到 $z_v=Wh_v$，再在这个 learned representation space 中计算 neighbor importance，同时 $Wh_u$ 也是后续被 attention 加权聚合的 message。

## 3. Step 2 — 计算未归一化 attention score $e_vu$

CS224W 课件先把 attention mechanism 写成一个一般函数：

$$
e_{vu}
=
a\left(
W^{(l)}h_u^{(l-1)},
W^{(l)}h_v^{(l-1)}
\right)
$$

令：

$$
z_u = W^{(l)}h_u^{(l-1)}, \qquad
z_v = W^{(l)}h_v^{(l-1)}
$$

那么：

$$
e_{vu}=a(z_u,z_v)
$$

其中：

- \(u\)：neighbor / sender
- \(v\)：target / receiver
- \($e_{vu}$\)：**node \(u\) 的 message 对 node \(v\) 的未归一化重要性分数**
- \($a(\cdot$)\)：attention mechanism，本身有 trainable parameters

课件给出的一个具体例子是：

$$
e_{vu}
=
Linear\left(
CONCAT(z_u,z_v)
\right)
$$

shape：

```text
z_u          [d_out]
z_v          [d_out]
   ↓ CONCAT
pair         [2 * d_out]
   ↓ Linear(2*d_out → 1)
e_vu         scalar
```

> 补充：经典 GAT / PyG 的实现会在这个线性 attention score 上使用 LeakyReLU，再进入 softmax。这里先按课件主线理解为 `attention mechanism a`。

## 必须回答 3

为什么 attention score 不能只看 `u`，而经典 GAT 要同时看 target `v` 和 neighbor `u`？

提示：同一个 neighbor `u` 对不同 target node，重要性未必一样。

你的回答：因为 attention score 要衡量的是“neighbor u 的 message 对当前 target node v 有多重要”。同一个 u 对不同的 target node 可能重要性不同，所以不能只根据 u 自己计算 score，而要同时考虑 u 和 v 的表示，从而建模它们之间的关系

## 4. Step 3 — Softmax over neighbors

固定一个 target node `v`：

```text
N(v) = {u1, u2, u3}
```

先得到：

```text
e_vu1
e_vu2
e_vu3
```

然后在 `v` 的邻居集合内部做：

$$
\alpha_{vu}
=
\frac{\exp(e_{vu})}
{\sum_{k\in N(v)\cup\{v\}}\exp(e_{vk})}
$$

于是：

$$
\sum_{u\in N(v)\cup\{v\}}
\alpha_{vu}=1
$$

注意：

> softmax 不是对全图所有 edge 一起做，而是对每个 target node 的 neighborhood 单独归一化。

In [2]:
scores = torch.tensor([1.0, 2.0, 0.0])
alpha = F.softmax(scores, dim=0)

print("scores:", scores)
print("alpha :", alpha)
print("sum   :", alpha.sum())

scores: tensor([1., 2., 0.])
alpha : tensor([0.2447, 0.6652, 0.0900])
sum   : tensor(1.)


## 必须回答 4

为什么 softmax 必须对同一个 target node `v` 的 neighbors 做，而不是对全图所有 edges 一起做？

你的回答：因为 attention weight 表示的是“对于固定 target node v，不同 neighbors 的相对重要性”。因此 softmax 应该只在 $N(v)$ 内归一化，使这些 neighbor attention weights 之和为 1。不同 target node 应该各自拥有独立的 attention distribution，不能让全图不同 target 的 edges 一起竞争。

## 5. Step 4 — Attention-weighted aggregation

拿到：

$$
\alpha_{vu}
$$

以后：

$$
h_v'
=
\sigma
\left(
\sum_{u\in N(v)\cup\{v\}}
\alpha_{vu} z_u
\right)
$$

其中：

$$
z_u = W h_u
$$

如果：

```text
3 neighbors
每个 z_u: [d_out]
每个 α_vu: scalar
```

那么：

```text
α_vu * z_u
→ [d_out]
```

所有 neighbors 再 sum：

```text
[num_neighbors, d_out]
      ↓ sum(dim=0)
[d_out]
```

## 必须回答 5

假设：

```text
neighbor_messages.shape = [5, 8]
attention_weights.shape = [5]
```

把 attention weight 乘到每个 neighbor message 后，再沿 neighbor dimension 求和，最终 shape 是什么？

你的回答：[8]

## 6. 一层 GAT 的完整链条

```text
neighbor u / target v
        ↓
shared W
        ↓
z_u, z_v
        ↓
attention mechanism a(z_u, z_v)
        ↓
e_vu
        ↓
softmax over N(v)
        ↓
α_vu
        ↓
α_vu * z_u
        ↓
sum over neighbors
        ↓
new h_v
```

课件主线公式：

$$
e_{vu}
=
a(z_u,z_v)
$$

$$
\alpha_{vu}
=
\frac{\exp(e_{vu})}
{\sum_{k\in N(v)}\exp(e_{vk})}
$$

$$
h_v^{(l)}
=
\sigma\left(
\sum_{u\in N(v)}
\alpha_{vu} z_u
\right)
$$

其中：

$$
z_u = W^{(l)}h_u^{(l-1)}
$$

## 7. 手搓教学版 single-head GAT layer

下面实现只为了把公式和代码对应起来：

- Python loop
- 不追求效率
- 真实项目用 PyG `GATConv`

In [19]:
class SimpleGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, negative_slope=0.2):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.attn = nn.Parameter(torch.empty(2*out_dim))
        self.negative_slope = negative_slope
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.xavier_uniform_(self.attn.view(1,-1))

    def forward(self, x, edge_index):
        z = self.linear(x)
        src, dst = edge_index
        
        out = []

        for v in range(x.shape[0]):
            mask = (dst == v)
            nbrs = src[mask]

            if not (nbrs == v).any():
                nbrs = torch.cat([
                    nbrs,
                    torch.tensor([v], device=x.device, dtype=torch.long)
                ])

            z_u = z[nbrs]
            z_v = z[v]

            z_v_repeat = z_v.unsqueeze(0).expand(z_u.shape[0],-1)

            pair = torch.cat([z_u, z_v_repeat], dim=1)

            e = F.leaky_relu(
                pair @ self.attn,
                negative_slope=self.negative_slope
            )

            alpha = F.softmax(e, dim=0)

            h_v = (alpha.unsqueeze(1) * z_u).sum(dim=0)

            out.append(h_v)

        return torch.stack(out,dim=0)       

In [20]:
edge_index = torch.tensor([
    [0, 1, 1, 2, 2, 3, 3, 4, 4, 5,
     1, 0, 2, 1, 3, 2, 4, 3, 5, 4],
    [1, 0, 2, 1, 3, 2, 4, 3, 5, 4,
     0, 1, 1, 2, 2, 3, 3, 4, 4, 5],
], dtype=torch.long)

gat = SimpleGATLayer(in_dim=3, out_dim=4)
H1 = gat(H, edge_index)

print("input :", H.shape)
print("output:", H1.shape)
print(H1)

input : torch.Size([6, 3])
output: torch.Size([6, 4])
tensor([[ 0.7947, -1.2234,  0.6421,  0.8198],
        [ 0.0584, -0.6833,  0.0579,  1.0369],
        [ 0.2385, -0.6909,  0.2899,  0.9442],
        [-0.1795, -0.3099,  0.0191,  1.0574],
        [ 0.3256, -0.7219,  0.3615,  0.8699],
        [ 0.5906, -0.8788,  0.6191,  0.8169]], grad_fn=<StackBackward0>)


## 必须回答 6

在：

```python
pair = torch.cat([z_u, z_v_repeat], dim=1)
```

假设：

```text
z_u.shape          = [3, 4]
z_v_repeat.shape   = [3, 4]
```

那么：

```text
pair.shape = ?
```

为什么 attention 参数需要接收：

```text
2 * out_dim
```

维输入？

你的回答：pair.shape = [3, 8]。因为 z_u.shape = [3,4]，z_v_repeat.shape = [3,4]，torch.cat(..., dim=1) 是沿特征维拼接，所以第一维的 3 不变，第二维从 4+4 变成 8。因此 attention 参数需要接收 2 * out_dim 维输入。

## 8. Multi-Head Attention

CS224W 课件给出的核心 motivation：

> **Multi-head attention 用于稳定 attention mechanism 的学习过程。**

做法是：

> 创建多个 attention mechanism，每个 head 有自己的一套参数，分别得到一组 attention scores。

例如第 \(k\) 个 head：

$$
h_v^{(l,k)}
=
\sigma\left(
\sum_{u\in N(v)}
\alpha_{vu}^{(k)}
W^{(l,k)}
h_u^{(l-1)}
\right)
$$

然后把多个 head 的输出再做 aggregation。

### 常见方式 1：CONCAT

如果每个 head 输出：

```text
[out_dim]
```

有 \(K\) 个 heads：

```text
K × [out_dim]
      ↓ concat
[K * out_dim]
```

写成：

$$
h_v^{(l)}
=
h_v^{(l,1)}
\Vert
h_v^{(l,2)}
\Vert
\cdots
\Vert
h_v^{(l,K)}
$$

### 常见方式 2：SUM / MEAN

课件写的是可以通过 **concatenation 或 summation** 聚合多个 heads。

在 PyG 里：

- `concat=True`：把 heads 拼接
- `concat=False`：对 heads 做平均，而不是拼接

所以 `concat=False` 时，feature dimension 不会乘上 `heads`。

## 必须回答 7

如果：

```text
out_channels = 8
heads = 4
concat = True
```

输出 feature dimension 是多少？

如果：

```text
concat = False
```

又是多少？

你的回答：concat = True的话，输出 feature dimension 是 8x4=32 ；concat = False的话，输出 feature dimension 是 8

## 9. PyG `GATConv`

典型调用：

```python
conv = GATConv(
    in_channels,
    out_channels,
    heads=4,
    concat=True
)

x = conv(x, edge_index)
```

内部会完成：

- feature transformation
- edge-wise attention score
- neighborhood softmax
- attention-weighted aggregation
- multi-head

In [21]:
try:
    from torch_geometric.nn import GATConv

    conv = GATConv(
        in_channels=3,
        out_channels=4,
        heads=2,
        concat=True
    )

    out = conv(H, edge_index)

    print("input shape :", H.shape)
    print("output shape:", out.shape)
    print("expected feature dim = 4 * 2 = 8")

except Exception as e:
    print("PyG GATConv 暂时无法运行：", repr(e))

D:\anaconda\envs\graph-learning\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


input shape : torch.Size([6, 3])
output shape: torch.Size([6, 8])
expected feature dim = 4 * 2 = 8


## 必须回答 8

在：

```python
GATConv(x, edge_index)
```

里面：

- `x` 提供什么？
- `edge_index` 提供什么？
- attention 的 trainable parameters 在哪里？
- 为什么所有 node 可以共享同一个 GAT layer？

你的回答：x 提供所有 node features；edge_index 提供图的连接关系，即哪些 node 之间可以进行 message passing；attention 的 trainable parameters 存在 GATConv layer 内部。所有 node 可以共享同一个 GAT layer，是因为 GAT 学习的是共享的 feature transformation $W$ 和 attention mechanism $a$，而不是为每个 node 单独学习一套参数。不同 node/edge 因为输入特征不同，最终仍然可以产生不同的 representation 和 attention score

## 10. GAT Attention 的几个关键性质

CS224W 课件在 GAT 部分最后总结了几个重要性质。

### 1. Key benefit：不同 neighbor 可以有不同的重要性

$$
\alpha_{vu}
$$

由 attention mechanism 学习，而不是完全由 degree 等结构规则预先指定。

### 2. Computationally efficient

- edge 上的 attention coefficients 可以并行计算
- node 上的 aggregation 也可以并行

### 3. Storage efficient

稀疏图只需要存储大约：

$$
O(|V|+|E|)
$$

规模的信息。

而且 trainable parameters 的数量不随着 graph size 线性增长。

### 4. Localized

GAT 只在：

$$
u\in N(v)
$$

的 local neighborhood 内做 attention，而不是默认让所有 nodes 两两 attention。

### 5. Inductive capability

attention mechanism 是：

> **shared edge-wise mechanism**

它依赖局部 sender / target 的表示，而不是依赖某个固定的 global graph structure。

所以同一套 attention rule 可以复用到新的 nodes / neighborhoods。

## 11. GAT vs Transformer Attention（补充对照）

> 这一小节是为了对应你的 Day13 计划中的 “Transformer 对照”。  
> 当前 PDF 在 GAT 页引用了 Vaswani et al.，但没有展开 Transformer attention 的完整推导，所以这里作为**补充理解**，不当作这份 PDF 的主讲内容。

共同点：

> 都会根据 pairwise relationship 得到 attention weight，再做 weighted aggregation。

| 方面 | GAT | Transformer |
|---|---|---|
| attention 范围 | 通常只在 graph neighbors 上 | vanilla self-attention 通常所有 token 两两交互 |
| connectivity | `edge_index` / graph structure 限制 | 通常形成 dense token-to-token attention |
| score | 经典 GAT 使用 learned attention mechanism $a(z_u,z_v)$ | scaled dot-product $QK^\top/\sqrt{d_k}$ |
| softmax | 对每个 target node 的 neighbors | 对每个 query 的 keys |
| aggregation value | transformed neighbor message \(z_u\) | value vector \(V_u\) |
| inductive bias | graph topology | token / sequence interactions |

最简洁的理解：

> **GAT = 在图结构约束下，对 local neighborhood 做 attention。**

## 必须回答 9

GAT 和 Transformer attention 最核心的相同点是什么？

最核心的不同点是什么？

你的回答：

相同点：GAT 和 Transformer 都通过 attention score + softmax 学习其他元素对当前元素的重要性，再根据 attention weights 对信息进行加权聚合。

不同点：GAT 的 attention 受到 graph topology 约束，只在当前 node 的 local neighborhood 中计算；标准 Transformer self-attention 通常在所有可见 tokens 之间计算。此外，经典 GAT 使用基于 $a^T[Wh_u\Vert Wh_v]$ 的 additive attention，而 Transformer 通常使用基于 $QK^T/\sqrt{d_k}$ 的 scaled dot-product attention。

## 12. KarateClub：PyG GAT Node Classification

目标：

```text
dataset
→ GAT
→ logits
→ train_mask loss
→ evaluation
```

In [22]:
try:
    import torch
    import torch.nn.functional as F
    from torch_geometric.datasets import KarateClub
    from torch_geometric.nn import GATConv

    dataset = KarateClub()
    data = dataset[0]

    print(data)
    print("num_features:", dataset.num_features)
    print("num_classes :", dataset.num_classes)

except Exception as e:
    print("KarateClub / PyG 暂时无法运行：", repr(e))

Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])
num_features: 34
num_classes : 4


In [30]:
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4):
        super().__init__()

        self.conv1 = GATConv(
            in_dim,
            hidden_dim,
            heads=heads,
            concat=True
        )

        self.conv2 = GATConv(
            hidden_dim * heads,
            out_dim,
            heads=1,
            concat=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)

        # 最后一层输出 logits
        x = self.conv2(x, edge_index)
        return x

In [39]:
class GAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=4):
        super().__init__()
        
        self.conv1 = GATConv(
            in_dim,
            hidden_dim,
            heads=heads,
            concat=True
        )

        self.conv2 = GATConv(
            hidden_dim*heads,
            out_dim,
            heads=1,
            concat=False
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.conv2(x, edge_index)

        return x

### shape 先搞清楚

假设：

```text
in_dim = 34
hidden_dim = 8
heads = 4
out_dim = 4
```

第一层：

```text
[34, 34]
↓ GATConv(34, 8, heads=4, concat=True)
[34, 32]
```

第二层：

```text
[34, 32]
↓ GATConv(32, 4, heads=1, concat=False)
[34, 4]
```

最终每个 node 得到 4 个 logits。

## 必须回答 10

为什么第二层输入是：

```text
hidden_dim * heads
```

而不是：

```text
hidden_dim
```

？

你的回答：因为第一层使用 concat=True，多个 attention heads 的输出会沿 feature dimension 拼接，所以第一层输出 feature dimension 为 hidden_dim * heads，因此第二层的 in_channels 必须是 hidden_dim * heads

In [40]:
def train_gat(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        logits = model(data.x, data.edge_index)

        loss = F.cross_entropy(
            logits[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            pred = logits.argmax(dim=1)
            acc = (pred == data.y).float().mean().item()

        history.append((loss.item(), acc))

        if epoch % 20 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch:03d} | "
                f"loss={loss.item():.4f} | "
                f"acc={acc:.4f}"
            )

    return history

In [41]:
try:
    gat_model = GAT(
        dataset.num_features,
        hidden_dim=8,
        out_dim=dataset.num_classes,
        heads=4
    )

    history = train_gat(
        gat_model,
        data,
        epochs=200
    )

except Exception as e:
    print("训练暂时无法运行：", repr(e))

Epoch 000 | loss=1.4194 | acc=0.4706
Epoch 020 | loss=0.5316 | acc=0.8824
Epoch 040 | loss=0.0307 | acc=0.7941
Epoch 060 | loss=0.0052 | acc=0.7941
Epoch 080 | loss=0.0035 | acc=0.7941
Epoch 100 | loss=0.0035 | acc=0.7941
Epoch 120 | loss=0.0036 | acc=0.7941
Epoch 140 | loss=0.0035 | acc=0.7941
Epoch 160 | loss=0.0033 | acc=0.7941
Epoch 180 | loss=0.0031 | acc=0.7941
Epoch 199 | loss=0.0026 | acc=0.7941


## 13. 观察 attention weights（可选，但推荐）

PyG 可以返回：

```text
edge u → v
↓
attention coefficient α_vu
```

In [42]:
try:
    gat_model.eval()

    with torch.no_grad():
        x1, attn_info = gat_model.conv1(
            data.x,
            data.edge_index,
            return_attention_weights=True
        )

    edge_idx_attn, alpha = attn_info

    print("edge_index shape:", edge_idx_attn.shape)
    print("alpha shape     :", alpha.shape)
    print()
    print("前 10 条 edge 的 attention weights：")
    print(alpha[:10])

except Exception as e:
    print("attention weight inspection 暂时无法运行：", repr(e))

edge_index shape: torch.Size([2, 190])
alpha shape     : torch.Size([190, 4])

前 10 条 edge 的 attention weights：
tensor([[0.0463, 0.0329, 0.0852, 0.0632],
        [0.0621, 0.0466, 0.0184, 0.0471],
        [0.0820, 0.0816, 0.1269, 0.0893],
        [0.1409, 0.0862, 0.1069, 0.1955],
        [0.1553, 0.0752, 0.1205, 0.1460],
        [0.1379, 0.0897, 0.1321, 0.1474],
        [0.1060, 0.1069, 0.1801, 0.1187],
        [0.0848, 0.0349, 0.0158, 0.0873],
        [0.1605, 0.0745, 0.1152, 0.1933],
        [0.3584, 0.4616, 0.4807, 0.5037]])


## 必须回答 11

如果：

```text
alpha.shape = [E, heads]
```

应该怎么理解这两个维度？

你的回答：第一维 $E$ 表示参与 message passing 的 edge 数量，每一行对应一条 $u\to v$ 的 edge；第二维 heads 表示 multi-head attention 中不同的 attention head，因此每条 edge 都会有 heads 个 attention coefficients。对于固定 target node $v$ 和固定 head，会在所有 incoming neighbors $u$ 之间进行 softmax，使这些 $\alpha_{vu}$ 之和为 1。

## 14. GCN / GraphSAGE / GAT 最终对照

| 模型 | neighbor aggregation 核心 | self 处理 | neighbor 权重 | 典型优势 |
|---|---|---|---|---|
| GCN | degree-normalized weighted sum | self-loop 一起聚合 | degree / graph rule | 简洁、经典 baseline |
| GraphSAGE | configurable AGG | self 与 neighbor summary 显式 combine | Mean 时近似等权，也可 Pool/LSTM | inductive + sampling + scalability |
| GAT | attention-weighted sum | 通常加入 self-loop | **由 feature 学出的 α** | 自适应学习 neighbor importance |

主线：

```text
GCN
固定 degree-based 权重
        ↓
GraphSAGE
可配置 aggregator + inductive / sampling
        ↓
GAT
学习每条 neighbor relation 的重要性
```

# Day 13 最终自测

1. 为什么 GAT 想把固定/结构决定的 neighbor weight 改成 learnable attention weight？
2. $z_v=W h_v$ 在做什么？shape 怎么变化？
3. $e_{vu}$ 是什么？为什么是 scalar？
4. 为什么 $e_{vu}$ 同时依赖 sender $u$ 和 target $v$？
5. 为什么 softmax 只在同一个 target node 的 neighborhood 内做？
6. $\alpha_{vu}$ 是什么？为什么对固定 $v$ 有：

$$
\sum_{u\in N(v)}\alpha_{vu}=1
$$

7. attention-weighted aggregation 的 shape 怎么走？
8. multi-head attention 的主要作用是什么？
9. `heads=4, out_channels=8, concat=True` 输出 feature dimension 是多少？
10. `concat=False` 时为什么输出维度不乘 `heads`？
11. GAT 为什么是 localized attention？
12. GAT 为什么具有 inductive capability？
13. `GATConv(x, edge_index)` 中：
    - data
    - graph structure
    - trainable rule  
    分别在哪里？
14. GAT 和 Transformer attention 的核心共同点 / 区别是什么？

---

## Day 13 通过标准

如果你能不看答案解释下面这条链，就算通过：

```text
h_u / h_v
↓
shared W
↓
z_u / z_v
↓
attention mechanism a(z_u, z_v)
↓
e_vu
↓
softmax over N(v)
↓
α_vu
↓
weighted neighbor aggregation
↓
new h_v
↓
multi-head
```

下一步：

> **Day 14 — Mini Project：同一数据集比较 MLP / GCN / GAT + embedding visualization + error analysis**